# N00｜亲自观察一次可回放故障诊断

**当前检查点：N00A 人工调查。** 你将面对一个固定的 Kubernetes 故障案例，只能通过缓存的只读探针逐步获取信息。不要先查看 `metadata.json`、`process-label` 或 `golden-trajectory`。

本文件是公开的空输出 starter，**不要直接运行**。在仓库根目录只执行一条命令：` .\.venv\Scripts\python.exe scripts\start_n00.py `。它会打开仓库外的工作副本，原始观察和答案不会保存进 Git。启动后的 readiness 单元会显示可点击的完整讲义路径。

## 1. 它在项目中的位置

```text
上游事故快照 → 只读探针 → 人工/Agent 调查轨迹 → N01 提炼评测缺口
```

今天不设计 Schema，也不实现本项目 Agent。今天的目标是先看到：输入有多不完整、探针怎样改变判断、最终答案为什么不足以代表调查质量。

In [ ]:
import json
import sys
from pathlib import Path
from IPython.display import Markdown, display

runtime_config_path = Path.cwd() / '.n00.runtime.json'
if not runtime_config_path.is_file():
    raise RuntimeError('请勿直接运行仓库中的 starter；请使用 scripts/start_n00.py 打开仓库外工作副本。')
runtime_config = json.loads(runtime_config_path.read_text(encoding='utf-8'))
PROJECT_ROOT = Path(runtime_config['project_root']).resolve()
TUTORIAL_ROOT = Path(runtime_config['tutorial_root']).resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))

from cloudopsbench_tutorial import (
    freeze_artifact,
    freeze_validated_orientation_comparison,
    load_pre_reveal_case,
    probe_catalog,
    replay_probe,
    reveal_comparison_material,
    search_probes,
    sha256_file,
    validate_orientation_report,
)

case = load_pre_reveal_case(TUTORIAL_ROOT)
print(json.dumps({k: case[k] for k in ('case_ref', 'namespace', 'symptom')}, ensure_ascii=False, indent=2))
lesson_path = PROJECT_ROOT / 'docs' / 'learning' / 'nodes' / 'N00_CLOUDOPSBENCH_ORIENTATION.md'
display(Markdown(f'[打开完整 N00 讲义]({lesson_path.as_uri()})'))
python_path = PROJECT_ROOT / '.venv' / 'Scripts' / 'python.exe'
runner_path = PROJECT_ROOT / 'scripts' / 'run_cloudopsbench_react.py'
print('N00B 稍后解锁；preflight 必须在终端运行：')
print(f'& "{python_path}" "{runner_path}" --preflight')

## 2. 学一点：这是调查，不是日志分类

你目前只知道一个高层症状。探针返回的是**观察**；只有当观察能支持或削弱某个假设时，它才成为你报告中的**证据**。Ground Truth 是教学案例的参考答案，在人工诊断和 Agent 轨迹都冻结以前不可见。

最近的替代做法是把全部日志和答案一次性给模型做分类。它启动更快，但无法观察工具选择、证据顺序、冗余调查或证据不足时是否拒答。

In [ ]:
# 先看 9 个工具族的用途与可用调用数量，不在 1680 个调用中盲猜。
for item in probe_catalog(TUTORIAL_ROOT):
    print(f"{item['tool_name']:<26} {item['available_calls']:>4}  {item['purpose']}")

## 3. 调查沙盒

先按工具族和参数关键词筛到 5～10 个候选，再选择一个 `probe_ref` 查看结果。高层症状通常先查告警、资源概况或依赖关系，而不是先把所有日志跑一遍。下面的搜索不会加载观察内容。建议总计调用 3～6 次；原始返回只存在仓库外工作副本，不复制进最终报告。

In [ ]:
TOOL_NAME = 'GetAlerts'  # 先从上面的 9 个工具族选一个
ARGUMENT_CONTAINS = ''   # 可用 namespace、资源类型或服务名继续缩小
candidates = search_probes(
    TUTORIAL_ROOT, tool_name=TOOL_NAME, argument_contains=ARGUMENT_CONTAINS
)
for item in candidates:
    print(item['ref'], json.dumps(item['arguments'], ensure_ascii=False))

In [ ]:
PROBE_REF = ''  # 在这里填入一个 probe-...，可重复修改并运行本单元
if PROBE_REF:
    result = replay_probe(TUTORIAL_ROOT, PROBE_REF)
    print(json.dumps(result, ensure_ascii=False, indent=2))
else:
    print('请选择一个 probe_ref。')

## 4. 唯一练习：冻结你的调查报告

先补全下面一个 TODO。每一步都写观察摘要、它怎样影响假设，以及为什么继续下一步；`limitations` 写当前仍不知道什么。冻结以后重新打开工作簿时，相同内容会显示“已冻结”，不会要求重做。

In [ ]:
# TODO：这是本节点唯一需要提交的学习者内容。
orientation_report = {
    'case_ref': case['case_ref'],
    'tutorial_manifest_sha256': sha256_file(TUTORIAL_ROOT / 'manifest.json').upper(),
    'symptom': case['symptom'],
    'initial_hypotheses': [
        # 至少两个在初始症状下仍可能成立的候选解释
    ],
    'steps': [
        # {
        #   'step': 1,
        #   'probe_ref': 'probe-...',
        #   'observation_summary': '我的摘要，不复制原始返回',
        #   'effect_on_hypothesis': '支持/削弱/无区分力，以及对象',
        #   'why_next': '下一步为什么这样查',
        # }
    ],
    'frozen_diagnosis': '',  # 根因判断，或 insufficient_evidence
    'limitations': [],
    'probe_value_assessment': {
        'useful': {'probe_ref': '', 'reason': ''},
        'low_value': {'probe_ref': '', 'reason': ''},
    },
}
errors = validate_orientation_report(TUTORIAL_ROOT, orientation_report)
print('通过 N00A 自检' if not errors else json.dumps(errors, ensure_ascii=False, indent=2))

In [ ]:
# 只有上一个单元没有错误时才运行一次。重复运行会拒绝覆盖已冻结证据。
errors = validate_orientation_report(TUTORIAL_ROOT, orientation_report)
if errors:
    raise ValueError(errors)
freeze_record = freeze_artifact(TUTORIAL_ROOT, 'human_diagnosis', orientation_report)
print('N00A 已冻结。请保存这个哈希：', freeze_record['sha256'])

## 5. N00B｜实际运行一次上游 ReAct Agent

当前没有 API key 时，状态应记录为 `N00-WAITING-KEY`，不是学习失败。准备好 OpenAI-compatible key 后，通过环境变量配置模型、endpoint、预算和密钥变量名。先做不调用模型的本地 preflight：

回到终端，复制 readiness 单元显示的绝对路径命令。

只有 preflight 通过后，才去掉 `--preflight` 执行一次真实运行。脚本不会修改上游 YAML；原始 trajectory 留在仓库外，只冻结白名单投影。Agent 答错也可以通过，只要真实调用、工具和终止轨迹有效。

## 6. N00C｜最后才揭晓答案

只有 `human_diagnosis` 和 `agent_projection` 两个哈希都存在且匹配时，下面单元才会加载 compare-only 材料。揭晓后，该案例永久不能进入 held-out、RAG 或性能指标。

In [ ]:
comparison_material = reveal_comparison_material(TUTORIAL_ROOT)
print(json.dumps(comparison_material, ensure_ascii=False, indent=2))
print('请比较人工与 Agent 的调查；工作簿位于仓库外，但仍不要把原始答案复制到公开摘要。')

### N00C completion checkpoint（现在不要填写）

这是 N00B 完成后才解锁的收尾表，不属于今天的 N00A。只写你自己的差异摘要，不复述 Ground Truth 原文。

In [ ]:
orientation_comparison = {
    'agreement_pattern': '',  # 人/Agent 与真值是一致、部分一致还是都不一致；不写真值原文
    'human_vs_agent_investigation': '',
    'most_discriminating_probe': '',
    'low_value_probe': '',
    'evaluator_gaps': [],  # N01 需要检查的 1～3 个问题
}
comparison_freeze = freeze_validated_orientation_comparison(
    TUTORIAL_ROOT, orientation_comparison
)
print('N00C 摘要已冻结：', comparison_freeze['sha256'])

## 7. 提交与通过

可执行工作簿始终位于仓库外；仓库中的 starter 保持空输出。N00 生成的报告、投影、比较、hash 与 marker 当前全部保持私有，不能复制进 Git；独立 public-export 门禁将在后续节点设计。

通过条件：

1. 人工报告字段完整，所有 probe 引用来自允许集合。
2. 人工诊断与 Agent 投影均在揭晓答案前冻结。
3. 能指出一个有价值 probe、一个低价值/冗余 probe，以及人工与 Agent 的一项调查差异。